# Portfolio construction and valuation

**Prerequisites:** complete **03** (market data and curves) and **05** (pricing fundamentals: `MarketContext`, instrument JSON, valuation flow) in this curriculum. This notebook builds a **portfolio spec** as JSON, wires **discounting** through a shared curve, and runs the **portfolio** parse → build → value → cashflow ladder pipeline.

## Concepts: portfolios as data, not objects

A **portfolio** here is a JSON **`PortfolioSpec`**: identifiers, base currency, **entities** (holders), and **positions**. Each position points at an **`instrument_spec`** (tagged JSON: `type` + `spec`)—the same shape you already use for single-instrument pricing.

The typical flow is:

1. **`parse_portfolio_spec`** — validate and canonicalize the spec string.
2. **`build_portfolio_from_spec`** — load into the engine and round-trip the spec (useful for debugging wire format).
3. **`value_portfolio`** — PV each position, convert to **base currency**, and return a **`PortfolioValuation`** JSON snapshot (per-position values, entity subtotals, **total** in base ccy).
4. **`aggregate_metrics`** — given that valuation JSON plus market and `as_of`, compute **portfolio-level metric aggregation** (summable risks, FX-aware where applicable).
5. **`aggregate_cashflows`** — merge contractual cashflows into a **date ladder** (`by_date`, `by_position`).

**Wire-format note:** `value_portfolio` returns **`PortfolioValuation`** only. Helpers **`portfolio_result_total_value`** and **`portfolio_result_get_metric`** expect a full **`PortfolioResult`** envelope (`valuation` + `metrics` + `meta`). The walkthrough shows both: read the total from the valuation JSON, then wrap a minimal `PortfolioResult` to exercise the helpers.

### Imports

Portfolio entry points live in `finstack.portfolio`; curves and **`MarketContext`** come from `finstack.core.market_data` (see notebook **03**).

In [1]:
import json
from datetime import date

from finstack.core.market_data import DiscountCurve, MarketContext
from finstack.portfolio import (
    aggregate_cashflows,
    aggregate_metrics,
    build_portfolio_from_spec,
    parse_portfolio_spec,
    portfolio_result_get_metric,
    portfolio_result_total_value,
    value_portfolio,
)

print("Imported portfolio pipeline and market helpers.")

Imported portfolio pipeline and market helpers.


### Market context: shared discount curve

Insert a **`DiscountCurve`** into **`MarketContext`**, then call **`to_json()`** so the same snapshot is passed into **`value_portfolio`**, **`aggregate_metrics`**, and **`aggregate_cashflows`**.

In [2]:
mc = MarketContext()
curve = DiscountCurve(
    "USD-OIS",
    date(2025, 1, 15),
    [
        (0.0, 1.0),
        (0.25, 0.9875),
        (0.5, 0.975),
        (1.0, 0.95),
        (2.0, 0.90),
        (5.0, 0.75),
    ],
    day_count="act_365f",
)
mc.insert(curve)
market_json = mc.to_json()
curves = json.loads(market_json).get("curves", [])
curve_ids = [c.get("id") for c in curves if isinstance(c, dict)]
print("Market OK; curve ids in snapshot:", curve_ids)

Market OK; curve ids in snapshot: ['USD-OIS']


### Portfolio spec: two USD deposits

Each **position** references **`discount_curve_id`** that must exist in **`market_json`**. **`quantity`** scales the economic exposure (here `1.0` **units**).

In [3]:
portfolio = json.dumps(
    {
        "id": "credit-portfolio",
        "as_of": "2025-01-15",
        "base_ccy": "USD",
        "entities": {"ENTITY-1": {"id": "ENTITY-1"}},
        "positions": [
            {
                "position_id": "POS-1",
                "entity_id": "ENTITY-1",
                "instrument_id": "DEP-3M",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-3M",
                        "notional": {"amount": 1_000_000.0, "currency": "USD"},
                        "start_date": "2025-01-15",
                        "maturity": "2025-04-15",
                        "day_count": "Act360",
                        "quote_rate": 0.045,
                        "discount_curve_id": "USD-OIS",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
            {
                "position_id": "POS-2",
                "entity_id": "ENTITY-1",
                "instrument_id": "DEP-6M",
                "instrument_spec": {
                    "type": "deposit",
                    "spec": {
                        "id": "DEP-6M",
                        "notional": {"amount": 2_000_000.0, "currency": "USD"},
                        "start_date": "2025-01-15",
                        "maturity": "2025-07-15",
                        "day_count": "Act360",
                        "quote_rate": 0.05,
                        "discount_curve_id": "USD-OIS",
                        "attributes": {},
                    },
                },
                "quantity": 1.0,
                "unit": "units",
            },
        ],
    }
)
spec_obj = json.loads(portfolio)
print("Spec id:", spec_obj["id"], "| positions:", len(spec_obj["positions"]))
print("Position ids:", [p["position_id"] for p in spec_obj["positions"]])

Spec id: credit-portfolio | positions: 2
Position ids: ['POS-1', 'POS-2']


### `parse_portfolio_spec`

Parse and canonicalize JSON. Failures raise **`ValueError`** with a Rust error string—fix the spec and retry.

In [4]:
canonical = parse_portfolio_spec(portfolio)
positions = json.loads(canonical).get("positions", [])
print("Parsed OK; canonical position count:", len(positions))
print("First instrument_id:", positions[0]["instrument_id"])

Parsed OK; canonical position count: 2
First instrument_id: DEP-3M


### `build_portfolio_from_spec`

Materialize the portfolio and return the spec after **`Portfolio::from_spec` → `to_spec`**. Use this to confirm instruments deserialize cleanly.

In [5]:
built = build_portfolio_from_spec(portfolio)
print("Round-trip prefix:", built[:200])
print("Full JSON length (chars):", len(built))

Round-trip prefix: {"id":"credit-portfolio","name":null,"base_ccy":"USD","as_of":"2025-01-15","entities":{"ENTITY-1":{"id":"ENTITY-1","name":null}},"positions":[{"position_id":"POS-1","entity_id":"ENTITY-1","instrument_
Full JSON length (chars): 393


### `value_portfolio`

Returns **`PortfolioValuation`** JSON. The **base-currency total** is under **`total_base_ccy`** (money object: `amount`, `currency`).

In [6]:
val_result = value_portfolio(portfolio, market_json)
val_obj = json.loads(val_result)
print("Valuation keys:", list(val_obj.keys()))
print("Snippet:\n", json.dumps(val_obj, indent=2)[:900], "...")
total_amt = float(val_obj["total_base_ccy"]["amount"])
total_ccy = val_obj["total_base_ccy"]["currency"]
print(f"Total PV (from valuation JSON): {total_amt:,.6f} {total_ccy}")

Valuation keys: ['as_of', 'position_values', 'total_base_ccy', 'by_entity']
Snippet:
 {
  "as_of": "2025-01-15",
  "position_values": {
    "POS-1": {
      "position_id": "POS-1",
      "entity_id": "ENTITY-1",
      "value_native": {
        "amount": "-1217.4565053925580286886543032",
        "currency": "USD"
      },
      "value_base": {
        "amount": "-1217.4565053925580286886543032",
        "currency": "USD"
      },
      "metric_scale": 1.0,
      "risk_metrics_complete": true
    },
    "POS-2": {
      "position_id": "POS-2",
      "entity_id": "ENTITY-1",
      "value_native": {
        "amount": "-557.8319573441235661448445169",
        "currency": "USD"
      },
      "value_base": {
        "amount": "-557.8319573441235661448445169",
        "currency": "USD"
      },
      "metric_scale": 1.0,
      "risk_metrics_complete": true
    }
  },
  "total_base_ccy": {
    "amount": "-1775.2884627366815948334988202",
    "currency": "USD"
  },
  "by_entity": ...
Total PV 

### `portfolio_result_total_value` and `portfolio_result_get_metric`

These read a **`PortfolioResult`** envelope. Below we wrap the valuation dict with minimal **`metrics`** and **`meta`** so the helpers have a valid document. In production, reporting layers may emit this full shape for you.

In [7]:
meta = {
    "numeric_mode": "F64",
    "rounding": {
        "mode": "Bankers",
        "ingest_scale_by_ccy": {},
        "output_scale_by_ccy": {},
        "version": 1,
        "tolerances": {"generic_epsilon": 1e-10, "rate_epsilon": 1e-12},
    },
}
pv_total = float(val_obj["total_base_ccy"]["amount"])
metrics_payload = {
    "aggregated": {
        "pv": {
            "metric_id": "pv",
            "total": pv_total,
            "by_entity": {"ENTITY-1": pv_total},
        }
    },
    "by_position": {},
}
portfolio_result_json = json.dumps(
    {"valuation": val_obj, "metrics": metrics_payload, "meta": meta}
)
print("portfolio_result_total_value:", portfolio_result_total_value(portfolio_result_json))
print("portfolio_result_get_metric('pv'):", portfolio_result_get_metric(portfolio_result_json, "pv"))
print("portfolio_result_get_metric('dv01'):", portfolio_result_get_metric(portfolio_result_json, "dv01"))

portfolio_result_total_value: -1775.2884627366816
portfolio_result_get_metric('pv'): -1775.2884627366816
portfolio_result_get_metric('dv01'): None


### `aggregate_metrics`

Pass the **`value_portfolio`** JSON string, base currency, market JSON, and **`as_of`**. *Current limitation:* the valuation JSON from **`value_portfolio`** does not rehydrate per-position risk measures, so aggregation from this round-trip may return **empty** `aggregated` / `by_position` until the wire format includes them. The call pattern below is still the supported API.

In [8]:
agg = aggregate_metrics(val_result, "USD", market_json, "2025-01-15")
agg_obj = json.loads(agg)
print(json.dumps(agg_obj, indent=2))
print("Aggregated metric ids:", list(agg_obj.get("aggregated", {}).keys()))

{
  "aggregated": {},
  "by_position": {}
}
Aggregated metric ids: []


### `aggregate_cashflows`

Returns **`by_date`** (ladder) and **`by_position`** schedules. Amounts serialize as decimal strings on the wire.

In [9]:
cfs = aggregate_cashflows(portfolio, market_json)
cf_obj = json.loads(cfs)
print(json.dumps(cf_obj, indent=2)[:1400])
print("... [truncated if long] ...")
print("Ladder dates:", sorted(cf_obj.get("by_date", {}).keys()))

{
  "by_date": {
    "2025-01-15": {
      "USD": {
        "amount": "-3000000",
        "currency": "USD"
      }
    },
    "2025-04-15": {
      "USD": {
        "amount": "1011249.9999999998835846781728",
        "currency": "USD"
      }
    },
    "2025-07-15": {
      "USD": {
        "amount": "2050277.7777777777519077062602",
        "currency": "USD"
      }
    }
  },
  "by_position": {
    "POS-1": [
      [
        "2025-01-15",
        {
          "amount": "-1000000",
          "currency": "USD"
        }
      ],
      [
        "2025-04-15",
        {
          "amount": "1011249.9999999998835846781728",
          "currency": "USD"
        }
      ]
    ],
    "POS-2": [
      [
        "2025-01-15",
        {
          "amount": "-2000000",
          "currency": "USD"
        }
      ],
      [
        "2025-07-15",
        {
          "amount": "2050277.7777777777519077062602",
          "currency": "USD"
        }
      ]
    ]
  },
  "position_summaries": {
    "P

## Mini-example: three deposits (3M, 6M, 1Y)

Same curve and `as_of`. Different notionals and quoted rates. We print **total PV**, **`aggregate_metrics`** (may be empty), and a readable **cashflow ladder** sorted by date.

In [10]:
def deposit_position(
    position_id: str,
    instrument_id: str,
    notional: float,
    quote_rate: float,
    maturity: str,
) -> dict:
    return {
        "position_id": position_id,
        "entity_id": "ENTITY-1",
        "instrument_id": instrument_id,
        "instrument_spec": {
            "type": "deposit",
            "spec": {
                "id": instrument_id,
                "notional": {"amount": notional, "currency": "USD"},
                "start_date": "2025-01-15",
                "maturity": maturity,
                "day_count": "Act360",
                "quote_rate": quote_rate,
                "discount_curve_id": "USD-OIS",
                "attributes": {},
            },
        },
        "quantity": 1.0,
        "unit": "units",
    }


three_deposit_portfolio = json.dumps(
    {
        "id": "rates-ladder-book",
        "as_of": "2025-01-15",
        "base_ccy": "USD",
        "entities": {"ENTITY-1": {"id": "ENTITY-1"}},
        "positions": [
            deposit_position("L-3M", "DEP-3M-ME", 1_000_000.0, 0.045, "2025-04-15"),
            deposit_position("L-6M", "DEP-6M-ME", 2_000_000.0, 0.050, "2025-07-15"),
            deposit_position("L-1Y", "DEP-1Y-ME", 1_500_000.0, 0.055, "2026-01-15"),
        ],
    }
)

mini_val = value_portfolio(three_deposit_portfolio, market_json)
mini_v = json.loads(mini_val)
mini_total = float(mini_v["total_base_ccy"]["amount"])
print(f"Three-deposit total PV: {mini_total:,.6f} USD")

mini_agg = aggregate_metrics(mini_val, "USD", market_json, "2025-01-15")
print("aggregate_metrics:", mini_agg)

mini_cf = json.loads(aggregate_cashflows(three_deposit_portfolio, market_json))
by_date = mini_cf.get("by_date", {})
print("Cashflow ladder (date -> USD amount):")
for d in sorted(by_date.keys()):
    usd = by_date[d].get("USD", {})
    amt = float(usd.get("amount", "0"))
    print(f"  {d}: {amt:,.2f} USD")
print("Done.")

Three-deposit total PV: 2,688.253204 USD
aggregate_metrics: {"aggregated":{},"by_position":{}}
Cashflow ladder (date -> USD amount):
  2025-01-15: -4,500,000.00 USD
  2025-04-15: 1,011,250.00 USD
  2025-07-15: 2,050,277.78 USD
  2026-01-15: 1,583,645.83 USD
Done.


## Takeaways

- A **portfolio** is **`PortfolioSpec` JSON**: entities + positions with embedded **`instrument_spec`** blocks.
- **`MarketContext.to_json()`** is the shared **market snapshot** for value, metrics, and cashflows.
- **`value_portfolio`** produces **`PortfolioValuation`** JSON; **total PV** is **`total_base_ccy.amount`** (parse with **`float`**).
- **`portfolio_result_*`** helpers expect a **`PortfolioResult`** envelope (**`valuation` + `metrics` + `meta`**).
- **`aggregate_cashflows`** gives a **date-indexed ladder** suitable for liquidity and P&L timing views.
- **`aggregate_metrics`** is the right hook for **summed risks** once valuation payloads carry per-position measures through JSON (or when called from native Rust with a live valuation).